# Cassava Leaf Disease Classifier — training + ONNX export

Trains an image classifier on the Kaggle **Cassava Leaf Disease Classification**
dataset (~26k images, 5 classes: CBB, CBSD, CGM, CMD, Healthy) and exports:

- `cassava_classifier.pt` — PyTorch checkpoint (state_dict + class map + config)
- `cassava_classifier.onnx` — ONNX export for CPU/edge inference
- `cassava_classifier.int8.onnx` — dynamically quantized INT8 ONNX (optional, faster CPU inference)

**Run this on Colab with a GPU runtime** (Runtime → Change runtime type → GPU).

You'll need a Kaggle account + API token (`kaggle.json`) to pull the dataset, and
you must have accepted the competition rules at
https://www.kaggle.com/c/cassava-leaf-disease-classification/rules first
(one click, required by Kaggle even though the data itself is free).


In [ ]:
# Install deps not preinstalled on Colab
!pip install -q timm onnx onnxruntime onnxscript kaggle


## 1. Get the dataset

Upload your `kaggle.json` (Kaggle → Account → Create New API Token) when prompted.


In [ ]:
from google.colab import files
import os, shutil

os.makedirs("/root/.kaggle", exist_ok=True)
uploaded = files.upload()  # select kaggle.json
for fname in uploaded:
    shutil.move(fname, f"/root/.kaggle/{fname}")
os.chmod("/root/.kaggle/kaggle.json", 0o600)


In [ ]:
DATA_DIR = "/content/cassava-data"
os.makedirs(DATA_DIR, exist_ok=True)

!kaggle competitions download -c cassava-leaf-disease-classification -p {DATA_DIR}
!cd {DATA_DIR} && unzip -q -o cassava-leaf-disease-classification.zip -d .
!ls {DATA_DIR}


## 2. Imports & config

In [ ]:
import json
import time
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import timm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
class CFG:
    data_dir = Path(DATA_DIR)
    img_dir = data_dir / "train_images"
    csv_path = data_dir / "train.csv"

    img_size = 380          # efficientnet_b0 default-ish; bump to 456+ if you want more accuracy
    batch_size = 32
    num_workers = 2
    epochs = 12
    lr = 3e-4
    weight_decay = 1e-4
    label_smoothing = 0.1
    val_frac = 0.15
    model_name = "efficientnet_b0"   # swap for "tf_efficientnetv2_s" if you want a bigger model
    num_classes = 5
    class_names = {
        0: "Cassava Bacterial Blight (CBB)",
        1: "Cassava Brown Streak Disease (CBSD)",
        2: "Cassava Green Mottle (CGM)",
        3: "Cassava Mosaic Disease (CMD)",
        4: "Healthy",
    }

cfg = CFG()


## 3. Explore the data

In [ ]:
df = pd.read_csv(cfg.csv_path)
print(df.shape)
print(df["label"].value_counts().sort_index())
df.head()


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for lbl in range(5):
    sample = df[df.label == lbl].sample(1, random_state=SEED).iloc[0]
    img = Image.open(cfg.img_dir / sample.image_id)
    axes[lbl].imshow(img)
    axes[lbl].set_title(cfg.class_names[lbl], fontsize=9)
    axes[lbl].axis("off")
plt.tight_layout()
plt.show()


## 4. Train / validation split (stratified)

In [ ]:
train_df, val_df = train_test_split(
    df, test_size=cfg.val_frac, stratify=df["label"], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f"train: {len(train_df)}  val: {len(val_df)}")


## 5. Dataset + transforms

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(cfg.img_size, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tfms = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class CassavaDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.img_dir / row.image_id).convert("RGB")
        img = self.transform(img)
        label = int(row.label)
        return img, label


train_ds = CassavaDataset(train_df, cfg.img_dir, train_tfms)
val_ds = CassavaDataset(val_df, cfg.img_dir, val_tfms)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                           num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                         num_workers=cfg.num_workers, pin_memory=True)


## 6. Model

In [ ]:
def build_model():
    model = timm.create_model(cfg.model_name, pretrained=True, num_classes=cfg.num_classes)
    return model.to(device)

model = build_model()
sum(p.numel() for p in model.parameters() if p.requires_grad)


## 7. Training loop

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, n = 0.0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits = model(imgs)
                loss = criterion(logits, labels)

            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        n += imgs.size(0)

    return total_loss / n, correct / n


In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_acc = 0.0
best_state = None

for epoch in range(cfg.epochs):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())

    print(f"epoch {epoch+1:02d}/{cfg.epochs}  "
          f"train_loss {tr_loss:.4f} train_acc {tr_acc:.4f}  "
          f"val_loss {val_loss:.4f} val_acc {val_acc:.4f}  "
          f"({time.time()-t0:.0f}s)")

print(f"best val acc: {best_acc:.4f}")
model.load_state_dict(best_state)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy"); axes[1].legend()
plt.tight_layout(); plt.show()


## 8. Evaluation

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        preds = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds,
                             target_names=[cfg.class_names[i] for i in range(cfg.num_classes)]))


In [ ]:
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(cfg.num_classes)); ax.set_yticks(range(cfg.num_classes))
ax.set_xticklabels(range(cfg.num_classes)); ax.set_yticklabels(range(cfg.num_classes))
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
for i in range(cfg.num_classes):
    for j in range(cfg.num_classes):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
plt.colorbar(im)
plt.tight_layout(); plt.show()


## 9. Save the PyTorch checkpoint (`.pt`)

In [ ]:
OUT_DIR = Path("/content/outputs")
OUT_DIR.mkdir(exist_ok=True)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "model_name": cfg.model_name,
    "img_size": cfg.img_size,
    "num_classes": cfg.num_classes,
    "class_names": cfg.class_names,
    "normalize_mean": IMAGENET_MEAN,
    "normalize_std": IMAGENET_STD,
    "best_val_acc": best_acc,
}

pt_path = OUT_DIR / "cassava_classifier.pt"
torch.save(checkpoint, pt_path)
print("saved", pt_path)


## 10. Export to ONNX

Exports with a dynamic batch axis so you can run batch-1 (drone frame-by-frame)
or batched inference. Then verifies PyTorch and ONNXRuntime agree.


In [ ]:
onnx_path = OUT_DIR / "cassava_classifier.onnx"

model.eval()
dummy_input = torch.randn(1, 3, cfg.img_size, cfg.img_size, device=device)

torch.onnx.export(
    model,
    dummy_input,
    str(onnx_path),
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    dynamo=False,  # use the legacy TorchScript-based exporter — avoids the onnxscript/dynamo path
)
print("saved", onnx_path)


In [ ]:
import onnx
import onnxruntime as ort

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)

ort_session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])

with torch.no_grad():
    torch_out = model(dummy_input).cpu().numpy()

ort_out = ort_session.run(None, {"input": dummy_input.cpu().numpy()})[0]

max_diff = np.abs(torch_out - ort_out).max()
print(f"max abs diff between PyTorch and ONNXRuntime outputs: {max_diff:.6f}")
assert max_diff < 1e-3, "ONNX export mismatch — investigate before trusting this model"
print("ONNX export verified ✓")


## 11. (Optional) INT8 dynamic quantization

Useful if this feeds the CPU-inference path of the drone pipeline. Dynamic
quantization is a one-liner and usually gives a solid CPU speedup with a small
accuracy hit — worth re-checking accuracy on the val set before relying on it.


In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

int8_path = OUT_DIR / "cassava_classifier.int8.onnx"
quantize_dynamic(
    model_input=str(onnx_path),
    model_output=str(int8_path),
    weight_type=QuantType.QInt8,
)
print("saved", int8_path)
print("fp32 size:", onnx_path.stat().st_size / 1e6, "MB")
print("int8 size:", int8_path.stat().st_size / 1e6, "MB")


In [ ]:
# Sanity-check accuracy of the quantized model on the val set
int8_session = ort.InferenceSession(str(int8_path), providers=["CPUExecutionProvider"])

correct, n = 0, 0
for imgs, labels in val_loader:
    out = int8_session.run(None, {"input": imgs.numpy()})[0]
    preds = out.argmax(1)
    correct += (preds == labels.numpy()).sum()
    n += len(labels)

print(f"INT8 ONNX val accuracy: {correct/n:.4f}  (fp32 was {best_acc:.4f})")


## 12. Download the artifacts

In [ ]:
from google.colab import files as colab_files

for f in [pt_path, onnx_path, int8_path]:
    colab_files.download(str(f))
